# Track 2b: Hidden Markov model regimes (Eshrit)

Follows `track2_regimes.ipynb`, where a GMM on the 15:30 market state passed the gate (`track2_gmm_probs_v1`,
dev log RMSE 0.45160). Most of that gain came from the index-event flags. The GMM added about 0.0005 on top.

**Why an HMM.** A GMM labels each day on its own. An HMM also learns that regimes *persist*: a stressed
market today makes a stressed market tomorrow more likely. So it should label borderline days more steadily,
using the recent path as well as today's snapshot.

**Hypothesis.** HMM regime probabilities beat GMM probabilities on high-stress stretches (2022, March 2023,
April 2025), because persistence smooths out one-day noise in the 15:30 state.

**What the team already found (notebook 16d).** Slow daily HMM regimes, from the previous day's data, made the
*daily* LightGBM worse. Here the question is different: whether regimes change how much the auction follows the
*15:30 surprise*.

| Variant | HMM inputs | Probabilities on date t use | Why |
|---|---|---|---|
| `track2_hmm_live_v1` | The six live 15:30 market-state numbers from track 2 (from 2021) | 15:30 state on dates up to and including t | Like-for-like with the GMM, plus persistence |
| `track2_hmm_slow_v1` | The team's four slow inputs from `regime_utils.py` (activity, realized vol, auction-share deviation, drawdown; from 2015) | Market data up to the close of t−1 | Does the slow macro regime change the intraday reaction? |

Both variants also get the three index-event flags, so they compare directly with `track2_gmm_probs_v1`.
K (2, 3 or 4 states) and the ridge strength are chosen on the validation year.

**Leakage controls.**
- Probabilities are *forward-filtered* (`regime_utils.filter_hmm`), never smoothed. hmmlearn's `predict_proba`
  uses later days, so it is not used.
- The HMM and its scaling are fitted only on dates before the rows they label. Tune and validation rows use an
  HMM fitted before the validation window; refit and test rows use one fitted before the test year.
- A no-look-ahead check in section 2 recomputes the probabilities with the future cut off.

In [1]:
import sys
from pathlib import Path
from functools import lru_cache
import numpy as np, pandas as pd
sys.path.insert(0, str(Path.cwd()))
from intraday_harness import Harness, leaderboard, RIDGE_GRID, REBALANCE, SEED, OUT, INCUMBENT
from notebooks.modeling_utils import fit_ridge, predict_ridge
from notebooks.regime_utils import fit_hmm, filter_hmm, build_regime_inputs, HMM_INPUTS

OWNER = "Eshrit"
QUICK = False                     # True = 2024 only, not recorded
K_GRID = (2, 3, 4)
RESTARTS = 10
h = Harness(quick=QUICK)

## 1. Inputs: live 15:30 market state (same as track 2) and the team's slow regime inputs

In [2]:
cols = ["date", "permno", "cp_available", "cp_volume_surprise20", "cp_trade_count_surprise20",
        "cp_recent_30m_volume_share", "cp_vwap_return_prior_close"]
panel = h.full_panel("2021-01-01", "2026-01-01", cols)
live = panel[panel.cp_available.astype(bool)].copy()
live["log_surprise"] = np.log(live.cp_volume_surprise20.clip(lower=1e-3))
live["log_trade_surprise"] = np.log(live.cp_trade_count_surprise20.clip(lower=1e-3))
live["abs_move"] = live.cp_vwap_return_prior_close.abs()
live["hot"] = live.log_surprise > np.log(1.5)
g = live.groupby("date")
state = pd.DataFrame({
    "mkt_pace": g.log_surprise.median(),
    "mkt_trade_pace": g.log_trade_surprise.median(),
    "mkt_dispersion": g.log_surprise.quantile(0.75) - g.log_surprise.quantile(0.25),
    "mkt_breadth": g.hot.mean(),
    "mkt_recent30": g.cp_recent_30m_volume_share.median(),
    "mkt_abs_move": g.abs_move.median(),
})
LIVE_COLS = list(state.columns)

reb = h.con.sql(f'''SELECT CAST(date AS DATE) AS date, max(russell_recon) AS mkt_russell_recon,
                           max(msci_review) AS mkt_msci_review, max(sp_quarterly_rebal) AS mkt_sp_rebal
                    FROM read_parquet('{REBALANCE}') GROUP BY 1''').df()
reb["date"] = pd.to_datetime(reb.date)
EVENT_COLS = ["mkt_russell_recon", "mkt_msci_review", "mkt_sp_rebal"]
events = reb.set_index("date")[EVENT_COLS]

# Slow inputs: each value on date t uses market data up to the close of t-1 (built by the team in regime_utils).
slow = build_regime_inputs()[HMM_INPUTS]
slow.index = pd.to_datetime(slow.index)
slow = slow.loc["2015-01-01":]
print(f"live state: {len(state)} dates from {state.index.min():%Y-%m-%d}; "
      f"slow inputs: {slow.dropna().shape[0]} complete dates from {slow.dropna().index.min():%Y-%m-%d}")

live state: 1245 dates from 2021-01-04; slow inputs: 2617 complete dates from 2015-08-06


## 2. HMM regime probabilities, forward-filtered

States are ordered by their mean of the first input (market pace for the live HMM, activity for the slow one),
so p0 is the quietest state.

In [3]:
SOURCES = {"live": (state, LIVE_COLS), "slow": (slow, HMM_INPUTS)}

@lru_cache(maxsize=None)
def hmm_probs(source: str, train_end: str, k: int):
    data, cols = SOURCES[source]
    train = data.loc[data.index < pd.Timestamp(train_end), cols].dropna()
    mu, sd = train.mean(), train.std()
    model, _ = fit_hmm(((train - mu) / sd).to_numpy(), k, restarts=RESTARTS, seed=SEED)
    z = ((data[cols] - mu) / sd).to_numpy()
    probs = filter_hmm(model, z)                     # P(state_t | inputs up to t): no later dates used
    order = np.argsort(model.means_[:, 0])
    frame = pd.DataFrame(probs[:, order], index=data.index, columns=[f"hmm_{source}{k}_p{j}" for j in range(k)])
    persistence = pd.Series(np.diag(model.transmat_)[order], name="stay_probability")
    return frame, persistence

def valid_start(year):
    return "2021-10-01" if year == 2022 else f"{year - 1}-01-01"

def hmm_features(frame, fold_year, role):
    train_end = valid_start(fold_year) if role in ("tune", "valid") else f"{fold_year}-01-01"
    out = events.reindex(frame.date).fillna(0).set_axis(frame.index)
    for source in SOURCES:
        for k in K_GRID:
            p, _ = hmm_probs(source, train_end, k)
            out = out.join(p.reindex(frame.date).set_axis(frame.index))
    return out

h.add_features(hmm_features)

# No-look-ahead check: refit on the same window, but filter a series cut off at a date; values up to it must match.
for source in SOURCES:
    data, cols = SOURCES[source]
    full, _ = hmm_probs(source, "2024-01-01", 3)
    cut = pd.Timestamp("2024-06-28")
    SOURCES[source] = (data.loc[:cut], cols)
    hmm_probs.cache_clear()
    truncated, _ = hmm_probs(source, "2024-01-01", 3)
    SOURCES[source] = (data, cols)
    hmm_probs.cache_clear()
    diff = (full.loc[:cut] - truncated).abs().max().max()
    print(f"{source}: max difference up to {cut:%Y-%m-%d} = {diff:.2e}", "PASS" if diff < 1e-9 else "FAIL")

live: max difference up to 2024-06-28 = 2.07e-12 PASS


slow: max difference up to 2024-06-28 = 1.12e-12 PASS


### How persistent are the regimes? (K = 3, fitted before 2024)

In [4]:
rows = []
for source in SOURCES:
    p, stay = hmm_probs(source, "2024-01-01", 3)
    top = p.loc[:"2023-12-31"].dropna().to_numpy().argmax(1)
    rows.append(pd.DataFrame({"source": source, "state": range(3), "stay_probability": stay.round(3),
                              "expected_run_days": (1 / (1 - stay)).round(1),
                              "share_of_days": np.bincount(top, minlength=3) / len(top)}))
pd.concat(rows, ignore_index=True)

,source,state,stay_probability,expected_run_days,share_of_days
0,live,0,0.865,7.4,0.280374
1,live,1,0.851,6.7,0.558077
2,live,2,0.661,2.9,0.161549
3,slow,0,0.964,27.6,0.401943
4,slow,1,0.940,16.6,0.346290
5,slow,2,0.975,39.8,0.251767


## 3. Fit, tune and submit

Same procedure as track 2: K and the ridge strength are chosen together on the validation year, then the model
is refitted through the year before the test year.

In [5]:
def cols_for(source, k): return [f"hmm_{source}{k}_p{j}" for j in range(k)] + EVENT_COLS

VARIANTS = {
    "track2_hmm_live_v1": {k: cols_for("live", k) for k in K_GRID},
    "track2_hmm_slow_v1": {k: cols_for("slow", k) for k in K_GRID},
}
DESCRIPTIONS = {
    "track2_hmm_live_v1": "Ridge + forward-filtered HMM regime probabilities on the live 15:30 market state (K on validation) + rebalance flags",
    "track2_hmm_slow_v1": "Ridge + forward-filtered HMM regime probabilities on the team's slow t-1 market inputs (K on validation) + rebalance flags",
}

def tune_and_fit(fold, options):
    z_tune, bounds = fold.target(fold.tune), fold.bounds(fold.tune)
    y_valid = fold.valid.log1p_auction_volume.to_numpy(float)
    best = None
    for k, extra in options.items():
        prep = fold.preprocessor(fold.tune, extra_continuous=extra)
        x_tune, x_valid = prep.transform(fold.tune), prep.transform(fold.valid)
        for strength in RIDGE_GRID:
            beta = fit_ridge(x_tune, z_tune, strength)
            forecast = fold.to_shares(predict_ridge(x_valid, beta), fold.valid, bounds)
            rmse = float(np.sqrt(np.mean((np.log1p(forecast) - y_valid) ** 2)))
            if best is None or rmse < best[0]:
                best = (rmse, k, strength)
    rmse, k, strength = best
    prep = fold.preprocessor(fold.refit, extra_continuous=options[k])
    beta = fit_ridge(prep.transform(fold.refit), fold.target(fold.refit), strength)
    return predict_ridge(prep.transform(fold.test), beta), {"k": k, "strength": strength, "valid_rmse": rmse}

chosen = []
for fold in h.folds():
    for name, options in VARIANTS.items():
        correction, settings = tune_and_fit(fold, options)
        h.submit(fold, correction, name=name, owner=OWNER, description=DESCRIPTIONS[name])
        chosen.append({"year": fold.year, "variant": name, **settings})
pd.DataFrame(chosen)

Model is not converging.  Current: -787.4762648726531 is not greater than -787.4734282237347. Delta is -0.0028366489184463717


Model is not converging.  Current: -821.294673227298 is not greater than -769.2289669500816. Delta is -52.06570627721635


Model is not converging.  Current: -822.2900993951166 is not greater than -822.2900523813121. Delta is -4.701380453298043e-05


Model is not converging.  Current: -4932.829962584242 is not greater than -4932.829902220076. Delta is -6.03641656198306e-05


track2_hmm_live_v1 2022: log RMSE 0.49421, WAPE 35.0331%, bias -14.75%


track2_hmm_slow_v1 2022: log RMSE 0.49408, WAPE 35.0050%, bias -14.76%


... (10 outputs left out)

Model is not converging.  Current: -7949.966060500573 is not greater than -7949.926244650753. Delta is -0.039815849820115545


track2_hmm_live_v1 2025: log RMSE 0.40395, WAPE 31.3798%, bias -5.70%


track2_hmm_slow_v1 2025: log RMSE 0.40398, WAPE 31.3727%, bias -5.76%


,year,variant,k,strength,valid_rmse
0,2022,track2_hmm_live_v1,4,0.0100,0.553927
1,2022,track2_hmm_slow_v1,2,0.0100,0.554532
2,2023,track2_hmm_live_v1,2,0.0100,0.493765
3,2023,track2_hmm_slow_v1,2,0.0100,0.494082
4,2024,track2_hmm_live_v1,3,0.0100,0.444189
5,2024,track2_hmm_slow_v1,3,0.0100,0.445784
6,2025,track2_hmm_live_v1,2,0.0001,0.414574
7,2025,track2_hmm_slow_v1,2,0.0001,0.415000


## 4. Results against the incumbent

In [6]:
results = {name: h.report(name) for name in VARIANTS}
pd.concat({n: r["summary"] for n, r in results.items()}).round(5)

development_rmse_log  \
                   model                                        
track2_hmm_live_v1 daily_anchor                       0.46403   
                   incumbent_tree_ridge               0.45445   
                   track2_hmm_live_v1                 0.45186   
track2_hmm_slow_v1 daily_anchor                       0.46403   
                   incumbent_tree_ridge               0.45445   
                   track2_hmm_slow_v1                 0.45242   

                                         development_wape  development_bias  \
                   model                                                      
track2_hmm_live_v1 daily_anchor                   0.34862          -0.15592   
                   incumbent_tree_ridge           0.33838          -0.13937   
                   track2_hmm_live_v1             0.33638          -0.13395   
track2_hmm_slow_v1 daily_anchor                   0.34862          -0.15592   
                   incumbent_tree_ridge           0.33838          -0.13937   
                   track2_hmm_slow_v1             0.33638          -0.13731   

                                         confirmation_rmse_log  \
                   model                                         
track2_hmm_live_v1 daily_anchor                        0.42314   
                   incumbent_tree_ridge                0.40613   
                   track2_hmm_live_v1                  0.40395   
track2_hmm_slow_v1 daily_anchor                        0.42314   
                   incumbent_tree_ridge                0.40613   
                   track2_hmm_slow_v1                  0.40398   

                                         confirmation_wape  
                   model                                    
track2_hmm_live_v1 daily_anchor                    0.32312  
                   incumbent_tree_ridge            0.31597  
                   track2_hmm_live_v1              0.31380  
track2_hmm_slow_v1 daily_anchor                    0.32312  
                   incumbent_tree_ridge            0.31597  
                   track2_hmm_slow_v1              0.31373

In [7]:
pd.DataFrame({n: r["gate"] for n, r in results.items()}) if not QUICK else "Gate is only applied on the full 2022-2025 run."

,track2_hmm_live_v1,track2_hmm_slow_v1
dev_rmse_gain,0.002595,0.00203
gain_ok,True,True
worst_year_change,-0.000973,-0.000353
annual_ok,True,True
wape_ok,True,True
interval_ok,True,True
passes_gate,True,True


### Head to head with the track 2 GMM and events-only versions

Paired block-bootstrap differences on identical rows (HMM minus the reference; negative means the HMM is better).
These need the full track 2 run's saved predictions.

In [8]:
refs = ["track2_gmm_probs_v1", "track2_events_only_v1"]
rows = []
for ref in refs:
    path = OUT / "submissions" / ref / "predictions.parquet"
    if not path.is_file():
        print(f"{ref}: run track2_regimes.ipynb in full first"); continue
    reference = pd.read_parquet(path)[["date", "permno", "forecast"]].rename(columns={"forecast": "ref"})
    reference = reference[reference.date.dt.year.isin(h.years)]
    for name in VARIANTS:
        cand = pd.concat(h._submissions[name])[["date", "permno", "auction_volume", "forecast"]].rename(columns={"forecast": "cand"})
        paired = cand.merge(reference, on=["date", "permno"], validate="one_to_one")
        iv = h._paired_intervals(paired, "ref", "cand").query("metric in ['rmse_log', 'wape']")
        rows.append(iv.assign(candidate=name, reference=ref))
pd.concat(rows)[["candidate", "reference", "period", "metric", "difference", "lower", "upper"]].round(5) if rows else None

,candidate,reference,period,metric,difference,lower,upper
0,track2_hmm_live_v1,track2_gmm_probs_v1,development,rmse_log,0.00025,-0.00017,0.00067
1,track2_hmm_live_v1,track2_gmm_probs_v1,development,wape,0.00023,-0.00012,0.00060
4,track2_hmm_live_v1,track2_gmm_probs_v1,confirmation,rmse_log,-0.00007,-0.00041,0.00025
5,track2_hmm_live_v1,track2_gmm_probs_v1,confirmation,wape,-0.00008,-0.00039,0.00022
0,track2_hmm_slow_v1,track2_gmm_probs_v1,development,rmse_log,0.00082,0.00023,0.00139
1,track2_hmm_slow_v1,track2_gmm_probs_v1,development,wape,0.00024,-0.00025,0.00070
4,track2_hmm_slow_v1,track2_gmm_probs_v1,confirmation,rmse_log,-0.00005,-0.00063,0.00055
5,track2_hmm_slow_v1,track2_gmm_probs_v1,confirmation,wape,-0.00015,-0.00098,0.00070
0,track2_hmm_live_v1,track2_events_only_v1,development,rmse_log,-0.00025,-0.00060,0.00008
1,track2_hmm_live_v1,track2_events_only_v1,development,wape,-0.00003,-0.00036,0.00030


Segment log RMSE (lower is better), with the track 2 GMM for comparison.

In [9]:
table = results[next(iter(VARIANTS))]["segments"][["daily_anchor", INCUMBENT]].join(
    pd.DataFrame({n: r["segments"][n] for n, r in results.items()}))
gmm_path = OUT / "submissions" / "track2_gmm_probs_v1" / "predictions.parquet"
if gmm_path.is_file():
    h._submissions["track2_gmm_probs_v1"] = [pd.read_parquet(gmm_path).query("date.dt.year in @h.years")]
    table = table.join(h.report("track2_gmm_probs_v1", record=False)["segments"][["track2_gmm_probs_v1"]])
table.join(table.drop(columns=["daily_anchor", INCUMBENT]).sub(table[INCUMBENT], axis=0).add_suffix(" vs incumbent")).round(4)

,daily_anchor,incumbent_tree_ridge,track2_hmm_live_v1,track2_hmm_slow_v1,track2_gmm_probs_v1,track2_hmm_live_v1 vs incumbent,track2_hmm_slow_v1 vs incumbent,track2_gmm_probs_v1 vs incumbent
segment,,,,,,,,
all,0.4656,0.4561,0.4536,0.4541,0.4533,-0.0025,-0.0019,-0.0027
fomc_day,0.4957,0.4796,0.4760,0.4741,0.4754,-0.0036,-0.0055,-0.0042
largest_1pct_auctions (diagnostic),1.1105,1.0681,1.0542,1.0558,1.0552,-0.0139,-0.0123,-0.0129
low_liquidity,0.5213,0.5234,0.5215,0.5221,0.5209,-0.0020,-0.0014,-0.0025
month_end,0.6563,0.6526,0.6370,0.6372,0.6363,-0.0156,-0.0154,-0.0162
msci_review,0.7786,0.7734,0.7135,0.7158,0.7149,-0.0600,-0.0576,-0.0586
quarterly_expiry,0.7610,0.7308,0.7302,0.7303,0.7314,-0.0006,-0.0005,0.0006
russell_recon,1.0273,1.0112,1.0226,1.0297,1.0194,0.0114,0.0185,0.0083
short_history,1.0171,0.9970,0.9957,0.9964,0.9956,-0.0013,-0.0007,-0.0015


In [10]:
leaderboard()[["name", "development_rmse_log", "development_wape", "confirmation_rmse_log", "dev_rmse_gain", "passes_gate"]]

,name,development_rmse_log,development_wape,confirmation_rmse_log,dev_rmse_gain,passes_gate
0,track2_gmm_interact_v1,0.451602,0.336080,0.404027,0.002850,True
1,track2_gmm_probs_v1,0.451603,0.336145,0.404022,0.002848,True
2,track2_hmm_live_v1,0.451856,0.336376,0.403953,0.002595,True
3,track2_events_only_v1,0.452110,0.336405,0.403971,0.002341,True
4,track2_hmm_slow_v1,0.452421,0.336382,0.403975,0.002030,True
5,track2_market_state_v1,0.452958,0.339121,0.405135,0.001493,False


## Findings (full run, 2022–2025)

| Variant | Dev log RMSE | Gain vs incumbent | Dev WAPE | 2025 log RMSE | Gate |
|---|---|---|---|---|---|
| Incumbent | 0.45445 | | 33.84% | 0.40613 | |
| `track2_events_only_v1` (track 2) | 0.45211 | 0.00234 | 33.64% | 0.40397 | pass |
| `track2_gmm_probs_v1` (track 2) | **0.45160** | **0.00285** | 33.61% | 0.40402 | pass |
| `track2_hmm_live_v1` | 0.45186 | 0.00260 | 33.64% | 0.40395 | pass |
| `track2_hmm_slow_v1` | 0.45242 | 0.00203 | 33.64% | 0.40398 | pass |

1. **The HMM does not beat the GMM.** The live HMM is 0.00025 worse than the GMM on development, with an
   interval from −0.00017 to +0.00067, so the two are statistically tied. The hypothesis that persistence would
   help in stressed stretches is not supported: 2022, the bear-market year, is where the HMM is slightly worse
   (0.49421 vs 0.49374).
2. **The slow macro HMM adds nothing beyond the event flags.** Against events only, the difference is +0.00031,
   with an interval from −0.00005 to +0.00066. It is clearly worse than the GMM (+0.00082, interval entirely
   above zero). This matches notebook 16d: slow market regimes don't help, for the daily model or the intraday
   correction.
3. **Why persistence doesn't help.** The live regimes are short-lived: the busy state lasts about 3 days on average.
   The useful information is today's 15:30 snapshot, which the GMM already uses. Smoothing over past days mostly
   adds lag.
4. The segments are almost identical to the GMM. The event-flag gains (MSCI −0.06, month-end −0.016) carry over,
   and Russell is still slightly worse.

**Verdict:** keep `track2_gmm_probs_v1` as track 2's candidate. The HMM is a clean negative result worth one
slide: we tested whether regime persistence matters, and it doesn't for the 15:30 correction.

*Note:* hmmlearn prints "Model is not converging" for a few restarts. These are tiny log-likelihood wobbles at the
stopping tolerance, and the best of 10 restarts is kept.
